# ⚾ KBO 기사 수집 데이터의 Snippet 추출 및 데이터베이스(SQLite) 연동 실습

본 노트북은 수집된 야구 뉴스 CSV 데이터(`아빌라_기사수집_260909_152613.csv`)를 바탕으로,
1. **CSV 데이터의 결측치(비어있는 Snippet) 상태 확인**
2. **기사 원문 URL로부터 기사 본문 요약(Snippet) 추출 및 보강**
3. **SQLite 데이터베이스 연결 및 `articles` 테이블 스키마 생성**
4. **보강된 데이터를 데이터베이스에 적재 (Insert or Ignore)**
5. **데이터베이스로부터 Snippet을 다각도로 조회 및 추출(SELECT)하는 SQL 쿼리 실습**
6. **추출한 Snippet 데이터를 AI 프롬프트 및 분석용으로 가공·활용하는 방법**

전 과정을 단계별로 다룹니다.

In [32]:
# 필요 라이브러리 임포트 (Python 기본 내장 sqlite3 사용 - 추가 설치 불필요)
import re
import time
import sqlite3
import requests
import pandas as pd
from bs4 import BeautifulSoup

print('라이브러리 로드 완료: pandas, sqlite3, requests, BeautifulSoup')

라이브러리 로드 완료: pandas, sqlite3, requests, BeautifulSoup


## 1. CSV 데이터 로드 및 Snippet 상태 점검
- 수집된 CSV 파일을 DataFrame으로 로드합니다.
- 현재 데이터에서 `snippet` 컬럼이 비어있는지(NaN) 확인합니다.

In [33]:
csv_path = '아빌라_기사수집_260909_152613.csv'
df = pd.read_csv(csv_path)

print(f'총 수집 기사 수: {len(df)}건')
print(f'데이터프레임 컬럼 목록: {list(df.columns)}')
print(f'snippet 컬럼 결측치(NaN) 수: {df["snippet"].isna().sum()}건 / 전체 {len(df)}건')

# 상위 5건 데이터 확인
df[['id', 'press', 'title', 'snippet']].head(5)

총 수집 기사 수: 16건
데이터프레임 컬럼 목록: ['id', 'category', 'title', 'press', 'date', 'url', 'snippet']
snippet 컬럼 결측치(NaN) 수: 16건 / 전체 16건


,id,press,title,snippet
0,1,스포츠동아,"대체 외인 성공 사례 SSG 아빌라, 공포의 투심으로 경계 대상 1순위",NaN
1,2,스포츠조선,화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...,NaN
2,3,스타뉴스,"'이럴수가' KBO 리그 9개 구단 초비상→'구단 최초 역사' 괴물 투수 ""한...",NaN
3,4,OSEN,"‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...",NaN
4,5,OSEN,"韓 9개 구단 초비상! 폰세 능가 외인, 생애 첫 완봉승→재계약 전격 요...",NaN


## 2. 기사 원문 URL로부터 본문 발췌(Snippet) 추출 및 데이터 보강
- CSV의 `snippet` 컬럼이 비어 있으므로, 각 행의 기사 `url`에 접속하여 본문 상위 150~200자를 추출하여 채워줍니다.
- **원칙 준수**: DataFrame의 컬럼 구성을 임의로 추가하거나 삭제하지 않고, 기존 `snippet` 컬럼의 결측치만 정제하여 채웁니다.
- **dtype 주의**: 전체 결측치인 컬럼은 pandas가 기본적으로 `float64`로 인식하므로, 문자열 삽입 전 `df['snippet'] = df['snippet'].astype(object)`로 타입을 변환합니다.

In [34]:
def extract_snippet_from_url(url: str, max_chars: int = 180) -> str:
    """네이버 스포츠 기사 상세 페이지 본문에서 앞부분 max_chars자를 요약문(snippet)으로 추출"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    }
    try:
        resp = requests.get(url, headers=headers, timeout=6)
        if resp.status_code != 200:
            return ''
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        # 최신 네이버 스포츠 본문 컨테이너 또는 레거시 본문 영역 탐색
        content_div = (
            soup.find('div', class_='_article_content')
            or soup.find('article', class_='_article_body')
            or soup.find('div', id='newsEndContents')
            or soup.find('div', class_=lambda c: c and ('news_end' in c or 'artice_body' in c))
        )
        if content_div:
            # 본문 텍스트 div는 보존하고 스크립트, 스타일, iframe, 버튼 등만 제거
            for tag in content_div(['script', 'style', 'iframe', 'button']):
                tag.decompose()
            text = content_div.get_text(separator=' ').strip()
            # 연속된 공백 및 줄바꿈 정리
            text = re.sub(r'\s+', ' ', text)
            return text[:max_chars].strip()
        return ''
    except Exception:
        return ''

print('기사 본문으로부터 Snippet 추출 및 보강 시작...')
# 결측치(NaN)로 인해 float64로 자동 추론된 컬럼 타입을 문자열 저장이 가능한 object 타입으로 변환
df['snippet'] = df['snippet'].astype(object)

# 전체 16건의 기사 URL에서 본문 요약문 추출
for idx, row in df.iterrows():
    if pd.isna(row['snippet']) or str(row['snippet']).strip() == '':
        snippet_text = extract_snippet_from_url(row['url'], max_chars=180)
        df.at[idx, 'snippet'] = snippet_text
        time.sleep(0.1)  # 네이버 서버 부하 방지 딜레이

print('Snippet 보강 완료!')
print(f'보강 후 snippet 유효 건수: {df["snippet"].str.len().gt(0).sum()}건')
df[['press', 'title', 'snippet']].head(5)

기사 본문으로부터 Snippet 추출 및 보강 시작...
Snippet 보강 완료!
보강 후 snippet 유효 건수: 16건


,press,title,snippet
0,스포츠동아,"대체 외인 성공 사례 SSG 아빌라, 공포의 투심으로 경계 대상 1순위",SSG 페드로 아빌라는 최근 연이은 호투로 9개 구단 경계 대상 1순위로 떠올랐다....
1,스포츠조선,화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...,광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@s...
2,스타뉴스,"'이럴수가' KBO 리그 9개 구단 초비상→'구단 최초 역사' 괴물 투수 ""한...",[스타뉴스 | 인천=김우종 기자] 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베...
3,OSEN,"‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...","SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런..."
4,OSEN,"韓 9개 구단 초비상! 폰세 능가 외인, 생애 첫 완봉승→재계약 전격 요...","SSG 랜더스 제공 [OSEN=인천, 이후광 기자] SSG 랜더스 를 제외한 프로야..."


## 3. SQLite 데이터베이스 생성 및 테이블 스키마 정의
- 경량 파일 기반 RDBMS인 **SQLite**를 사용합니다 (`baseball_news.db`).
- `articles` 테이블을 생성하며, 동일한 기사가 중복 적재되지 않도록 `url` 컬럼에 `UNIQUE` 제약조건을 부여합니다.
- 검색 속도 향상을 위해 날짜(`date`)와 언론사(`press`)에 인덱스를 생성합니다.

In [35]:
db_name = 'baseball_news.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# 기사 테이블 스키마 정의 (DDL)
create_table_sql = """
CREATE TABLE IF NOT EXISTS articles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    category TEXT,
    title TEXT NOT NULL,
    press TEXT,
    date TEXT,
    url TEXT UNIQUE,
    snippet TEXT,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""
cursor.execute(create_table_sql)
cursor.execute('CREATE INDEX IF NOT EXISTS idx_articles_date ON articles(date);')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_articles_press ON articles(press);')
conn.commit()

print(f"데이터베이스 파일 '{db_name}' 및 'articles' 테이블 스키마 생성 완료!")

데이터베이스 파일 'baseball_news.db' 및 'articles' 테이블 스키마 생성 완료!


## 4. 보강된 기사 데이터를 데이터베이스(DB)로 적재 (UPSERT)
- `ON CONFLICT(url) DO UPDATE` 구문(UPSERT)을 사용하여 이미 존재하는 기사 URL인 경우 비어있던 `snippet` 등의 정보를 최신 상태로 갱신하고, 새로운 기사는 신규 삽입합니다.
- 이를 통해 중복 수집 없이도 결측치 보강 작업이 안전하게 반영됩니다.

In [36]:
insert_sql = """
INSERT INTO articles (category, title, press, date, url, snippet)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(url) DO UPDATE SET
    snippet = excluded.snippet,
    category = excluded.category,
    title = excluded.title,
    press = excluded.press,
    date = excluded.date;
"""

inserted_count = 0
for _, row in df.iterrows():
    cursor.execute(insert_sql, (
        row['category'],
        row['title'],
        row['press'],
        row['date'],
        row['url'],
        row['snippet']
    ))
    inserted_count += 1

conn.commit()
print(f"DB 적재 및 갱신 완료: 총 {inserted_count}건의 기사 처리 완료")

# DB에 실제로 저장된 총 행 수 및 snippet 유효 건수 확인
cursor.execute('SELECT COUNT(*), COUNT(snippet) FROM articles WHERE snippet IS NOT NULL AND snippet != ""')
valid_snippet_count = cursor.fetchone()[0]
cursor.execute('SELECT COUNT(*) FROM articles')
total_db_count = cursor.fetchone()[0]
print(f"현재 DB 내 전체 기사 수: {total_db_count}건 (유효 Snippet 보유: {valid_snippet_count}건)")

DB 적재 및 갱신 완료: 총 16건의 기사 처리 완료
현재 DB 내 전체 기사 수: 16건 (유효 Snippet 보유: 16건)


## 5. 데이터베이스에서 Snippet 데이터 추출하기 (핵심 쿼리)
데이터베이스에 저장된 데이터로부터 다양한 분석 및 응용 목적에 맞게 `snippet`을 SQL 쿼리로 추출합니다.

### 5-1. 기본 추출: Snippet이 비어있지 않은 기사 전체 조회

In [37]:
query_all_snippets = """
SELECT id, press, title, snippet
FROM articles
WHERE snippet IS NOT NULL AND snippet != ''
ORDER BY id ASC
LIMIT 5;
"""

df_result_all = pd.read_sql_query(query_all_snippets, conn)
print('=== [추출 결과 1] 기본 Snippet 조회 (상위 5건) ===')
df_result_all

=== [추출 결과 1] 기본 Snippet 조회 (상위 5건) ===


,id,press,title,snippet
0,1,스포츠동아,"대체 외인 성공 사례 SSG 아빌라, 공포의 투심으로 경계 대상 1순위",SSG 페드로 아빌라는 최근 연이은 호투로 9개 구단 경계 대상 1순위로 떠올랐다....
1,2,스포츠조선,화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...,광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@s...
2,3,스타뉴스,"'이럴수가' KBO 리그 9개 구단 초비상→'구단 최초 역사' 괴물 투수 ""한...",[스타뉴스 | 인천=김우종 기자] 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베...
3,4,OSEN,"‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...","SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런..."
4,5,OSEN,"韓 9개 구단 초비상! 폰세 능가 외인, 생애 첫 완봉승→재계약 전격 요...","SSG 랜더스 제공 [OSEN=인천, 이후광 기자] SSG 랜더스 를 제외한 프로야..."


### 5-2. 조건부 추출: Snippet 본문에 특정 키워드('완봉', '투심', '에이스')가 포함된 기사만 선별
- SQL의 `LIKE '%키워드%'` 조건을 통해 특정 단어가 언급된 snippet만 정밀하게 필터링하여 추출합니다.

In [38]:
target_word = '완봉'
query_keyword = f"""
SELECT id, press, title, snippet
FROM articles
WHERE snippet LIKE '%{target_word}%'
"""

df_keyword = pd.read_sql_query(query_keyword, conn)
print(f"=== [추출 결과 2] Snippet 내 '{target_word}' 키워드 포함 기사: 총 {len(df_keyword)}건 ===")

for _, row in df_keyword.head(3).iterrows():
    print(f"[{row['press']}] {row['title']}")
    print(f"   발췌(Snippet): {row['snippet']}")
    print('-' * 80)

=== [추출 결과 2] Snippet 내 '완봉' 키워드 포함 기사: 총 11건 ===
[스포츠조선] 화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...
   발췌(Snippet): 광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@sportschosun.com [스포츠조선 고재완 기자] 페드로 아빌라 (29)가 지난 4일 인천 두산 베어스 전에서 101구 3안타 완봉승을 수확하며 재계약의 확실한 도장을 찍은 반면, 대체 외인 토머스 해치(32)는 어깨 통증이 가라앉지
--------------------------------------------------------------------------------
[OSEN] ‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...
   발췌(Snippet): SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런 괴물 외국인투수를 데려온 걸까. 연봉 5억 원을 받는 대체 외국인투수가 SSG 랜더스 창단 첫 완봉승의 주인공으로 우뚝 섰다. 프로야구 SSG 랜더스는 4일 인천SSG랜더스필드에서 열린 2026 신한 SOL KBO리그 두산 베어스와의 시즌
--------------------------------------------------------------------------------
[OSEN] 韓 9개 구단 초비상! 폰세 능가 외인, 생애 첫 완봉승→재계약 전격 요...
   발췌(Snippet): SSG 랜더스 제공 [OSEN=인천, 이후광 기자] SSG 랜더스 를 제외한 프로야구 9개 구단에 비상이 걸렸다. 지난해 정규시즌 MVP 코디 폰세(토론토 블루제이스)를 능가하는 괴물 외국인투수가 생애 첫 완봉승을 달성한 뒤 앞으로 2년 더 SSG에서 뛰고 싶다는 뜻을 밝혔다. SSG 외국인투수 페드로 아빌라 는 지난 4
-------------------------------------------------

### 5-3. 집계 및 통계 추출: 언론사별 Snippet 수 및 평균 글자 수

In [39]:
query_stats = """
SELECT 
    press,
    COUNT(*) AS article_count,
    ROUND(AVG(LENGTH(snippet)), 1) AS avg_snippet_length
FROM articles
WHERE snippet IS NOT NULL AND snippet != ''
GROUP BY press
ORDER BY article_count DESC;
"""

df_stats = pd.read_sql_query(query_stats, conn)
print('=== [추출 결과 3] 언론사별 Snippet 집계 현황 ===')
df_stats.head(10)

=== [추출 결과 3] 언론사별 Snippet 집계 현황 ===


,press,article_count,avg_snippet_length
0,OSEN,2,180.0
1,연합뉴스,2,174.0
2,뉴스1,1,180.0
3,뉴시스,1,180.0
4,마이데일리,1,180.0
5,스타뉴스,1,180.0
6,스포츠경향,1,179.0
7,스포츠동아,1,180.0
8,스포츠조선,1,179.0
9,엑스포츠뉴스,1,180.0


## 6. 추출된 Snippet 데이터의 실전 활용 (AI 프롬프트/분석 코퍼스 변환)
- DB에서 추출한 `snippet` 데이터를 리스트나 딕셔너리로 순회하여, AI 요약 보고서 작성용 입력 텍스트 포맷으로 가공합니다.

In [40]:
# DB에서 전체 snippet 목록 추출
cursor.execute('SELECT date, press, title, snippet FROM articles WHERE snippet IS NOT NULL AND snippet != ""')
article_records = cursor.fetchall()

# AI 분석 보고서에 입력할 구조화된 텍스트로 결합
briefing_snippets = []
for date, press, title, snip in article_records[:5]:
    briefing_snippets.append(f"- [{date} | {press}] {title}\n  ▶ 요약 발췌: {snip}")

final_corpus = "\n\n".join(briefing_snippets)
print('=== [실전 활용] DB에서 추출된 Snippet 기반 AI 분석용 프롬프트 데이터 ===\n')
print(final_corpus)

# 작업 완료 후 데이터베이스 연결 닫기
conn.close()
print('\n데이터베이스 연결이 안전하게 종료되었습니다.')

=== [실전 활용] DB에서 추출된 Snippet 기반 AI 분석용 프롬프트 데이터 ===

- [2026-09-09 | 스포츠동아] 대체 외인 성공 사례 SSG 아빌라, 공포의 투심으로 경계 대상 1순위
  ▶ 요약 발췌: SSG 페드로 아빌라는 최근 연이은 호투로 9개 구단 경계 대상 1순위로 떠올랐다. 가을야구 순위 싸움이 치열한 팀들로서는 아빌라와 맞대결이 유독 더 신경 쓰일 수밖에 없다. 사진제공｜SSG 랜더스 [스포츠동아 장은상 기자] SSG 랜더스 외국인 투수 페드로 아빌라 (29)가 경계 대상 1순위로 급부상했다. 아빌라는 SS

- [2026-09-09 | 스포츠조선] 화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...
  ▶ 요약 발췌: 광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@sportschosun.com [스포츠조선 고재완 기자] 페드로 아빌라 (29)가 지난 4일 인천 두산 베어스 전에서 101구 3안타 완봉승을 수확하며 재계약의 확실한 도장을 찍은 반면, 대체 외인 토머스 해치(32)는 어깨 통증이 가라앉지

- [2026-09-09 | 스타뉴스] '이럴수가' KBO 리그 9개 구단 초비상→'구단 최초 역사' 괴물 투수 "한...
  ▶ 요약 발췌: [스타뉴스 | 인천=김우종 기자] 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베어스와 홈 경기에서 SSG 랜더스 외국인 투수 페드로 아빌라의 활약 모습. /사진=SSG 랜더스 제공 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베어스와 홈 경기에서 SSG 랜더스 외국인 투수 페드로 아빌라의 활약 모습. 경기 후 취재진

- [2026-09-09 | OSEN] ‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...
  ▶ 요약 발췌: SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런 괴물 외국인투수를 데려온 걸까. 연봉 5억 원을 받는 대체 외국인투수가 SSG 랜더스 창단 

## 7. 키워드 정규화(Keyword Normalization) 및 동의어 통합 시스템 구현

동일한 대상을 지칭하지만 키워드 표기 차이(예: `한화` vs `한화이글스`, `KIA` vs `기아`)로 인해 발생하는
1. **DB 파일 분산 저장(파편화)** 현상 방지
2. **AI 보고서 핵심 기사 연관도 산출(`get_relevance_score`) 누락** 방지
3. **챗봇 RAG 검색 정합성 저하** 방지

를 위해 **KBO 도메인 사전 규칙**과 **OpenAI `gpt-5.6-luna` 지능형 Fallback**을 결합한 키워드 정규화 모듈을 구현하고 검증합니다.

In [1]:
# ============================================================
# 1. 키워드 정규화 엔진 (KeywordNormalizer) 구현
# ============================================================
import os
import re
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

load_dotenv(override=True)

@dataclass
class NormalizedEntity:
    raw: str                  # 사용자가 입력한 원본 문자열
    canonical: str            # 화면 및 보고서 제목용 대표 공식 명칭 (예: '한화 이글스')
    safe_id: str              # DB 파일명 및 시스템 식별자용 명칭 (예: '한화이글스')
    synonyms: List[str]       # 동의어 및 축약어 풀 (연관도 검색 및 RAG 매칭용)
    entity_type: str          # 'team'(구단), 'player'(선수), 'topic'(야구주제), 'general'(일반)

class KeywordNormalizer:
    """
    KBO 야구 뉴스 수집, AI 보고서, DB 적재를 위한 하이브리드 키워드 정규화기
    - 1차: KBO 10개 구단 및 야구 용어 사전 기반 고속 매핑 (Zero Latency)
    - 2차: 텍스트 전처리 (선수명 띄어쓰기 결합, 불필요한 조사 제거)
    - 3차: OpenAI gpt-5.6-luna 지능형 Fallback & 캐싱
    """

    # KBO 10개 구단 공식 명칭 및 동의어 매핑 테이블
    KBO_TEAMS = {
        "한화이글스": {
            "canonical": "한화 이글스",
            "safe_id": "한화이글스",
            "synonyms": ["한화", "한화이글스", "한화 이글스", "이글스"],
            "entity_type": "team",
        },
        "KIA타이거즈": {
            "canonical": "KIA 타이거즈",
            "safe_id": "KIA타이거즈",
            "synonyms": ["KIA", "기아", "기아타이거즈", "KIA타이거즈", "기아 타이거즈", "KIA 타이거즈", "타이거즈"],
            "entity_type": "team",
        },
        "LG트윈스": {
            "canonical": "LG 트윈스",
            "safe_id": "LG트윈스",
            "synonyms": ["LG", "엘지", "LG트윈스", "엘지트윈스", "LG 트윈스", "엘지 트윈스", "트윈스"],
            "entity_type": "team",
        },
        "SSG랜더스": {
            "canonical": "SSG 랜더스",
            "safe_id": "SSG랜더스",
            "synonyms": ["SSG", "쓱", "SSG랜더스", "SSG 랜더스", "랜더스", "SK와이번스", "SK 와이번스"],
            "entity_type": "team",
        },
        "두산베어스": {
            "canonical": "두산 베어스",
            "safe_id": "두산베어스",
            "synonyms": ["두산", "두산베어스", "두산 베어스", "베어스"],
            "entity_type": "team",
        },
        "삼성라이온즈": {
            "canonical": "삼성 라이온즈",
            "safe_id": "삼성라이온즈",
            "synonyms": ["삼성", "삼성라이온즈", "삼성 라이온즈", "라이온즈"],
            "entity_type": "team",
        },
        "롯데자이언츠": {
            "canonical": "롯데 자이언츠",
            "safe_id": "롯데자이언츠",
            "synonyms": ["롯데", "롯데자이언츠", "롯데 자이언츠", "자이언츠"],
            "entity_type": "team",
        },
        "KT위즈": {
            "canonical": "KT 위즈",
            "safe_id": "KT위즈",
            "synonyms": ["KT", "케이티", "KT위즈", "케이티위즈", "KT 위즈", "위즈"],
            "entity_type": "team",
        },
        "NC다이노스": {
            "canonical": "NC 다이노스",
            "safe_id": "NC다이노스",
            "synonyms": ["NC", "엔씨", "NC다이노스", "엔씨다이노스", "NC 다이노스", "다이노스"],
            "entity_type": "team",
        },
        "키움히어로즈": {
            "canonical": "키움 히어로즈",
            "safe_id": "키움히어로즈",
            "synonyms": ["키움", "키움히어로즈", "키움 히어로즈", "히어로즈", "넥센", "넥센히어로즈"],
            "entity_type": "team",
        },
    }

    # 주요 야구 일반 토픽 사전
    KBO_TOPICS = {
        "가을야구": ["포스트시즌", "가을야구", "PS", "준플레이오프", "플레이오프", "한국시리즈"],
        "FA": ["FA", "자유계약", "자유계약선수", "프리에이전트"],
        "신인드래프트": ["드래프트", "신인드래프트", "신인선수지명"],
    }

    def __init__(self, client: Optional[OpenAI] = None, model: str = "gpt-5.6-luna"):
        api_key = os.getenv("OPENAI_API_KEY")
        self.client = client or (OpenAI(api_key=api_key) if api_key else None)
        self.model = model
        self._cache: Dict[str, NormalizedEntity] = {}

        # 빠른 역방향 매핑 색인 (소문자/공백제거 -> 대상 키)
        self._lookup: Dict[str, str] = {}
        for team_key, info in self.KBO_TEAMS.items():
            for syn in info["synonyms"]:
                clean_syn = self._clean_token(syn)
                self._lookup[clean_syn] = team_key

    @staticmethod
    def _clean_token(text: str) -> str:
        return re.sub(r"\s+", "", text.lower())

    def _strip_josa(self, text: str) -> str:
        """불필요한 한국어 조사(은/는/이/가/의/을/를/과/와/에게/에서/도) 제거"""
        josa_pattern = r"(은|는|이|가|의|을|를|과|와|에게|에서|도|에)$"
        if len(text) > 2:
            return re.sub(josa_pattern, "", text).strip()
        return text.strip()

    def normalize(self, keyword: str, use_llm_fallback: bool = True) -> NormalizedEntity:
        """
        입력 키워드를 표준 엔티티(NormalizedEntity)로 변환합니다.
        """
        raw = keyword.strip()
        if not raw:
            return NormalizedEntity(raw="", canonical="야구", safe_id="야구", synonyms=["야구"], entity_type="general")

        # 캐시 확인
        if raw in self._cache:
            return self._cache[raw]

        # 1. 텍스트 기본 정제 및 조사 제거
        cleaned = self._strip_josa(raw)
        lookup_key = self._clean_token(cleaned)

        # 2. KBO 10개 구단 사전 검사
        if lookup_key in self._lookup:
            team_key = self._lookup[lookup_key]
            info = self.KBO_TEAMS[team_key]
            entity = NormalizedEntity(
                raw=raw,
                canonical=info["canonical"],
                safe_id=info["safe_id"],
                synonyms=list(set(info["synonyms"] + [raw, cleaned])),
                entity_type=info["entity_type"],
            )
            self._cache[raw] = entity
            return entity

        # 3. KBO 주요 토픽 사전 검사
        for topic_name, syn_list in self.KBO_TOPICS.items():
            for s in syn_list:
                if self._clean_token(s) == lookup_key:
                    entity = NormalizedEntity(
                        raw=raw,
                        canonical=topic_name,
                        safe_id=topic_name,
                        synonyms=list(set(syn_list + [raw, cleaned])),
                        entity_type="topic",
                    )
                    self._cache[raw] = entity
                    return entity

        # 4. 한국어 인명/선수명 형태 감지 (예: '김 도 영' -> '김도영')
        if re.fullmatch(r"^[가-힣]\s+[가-힣](\s+[가-힣])?$", cleaned):
            combined_name = re.sub(r"\s+", "", cleaned)
            entity = NormalizedEntity(
                raw=raw,
                canonical=combined_name,
                safe_id=combined_name,
                synonyms=list(set([raw, cleaned, combined_name])),
                entity_type="player",
            )
            self._cache[raw] = entity
            return entity

        # 5. LLM Fallback (OpenAI gpt-5.6-luna) - 사전에 없는 별명, 오타, 복합 표현 보정
        if use_llm_fallback and self.client:
            try:
                prompt = f"""당신은 한국 프로야구(KBO) 전문 데이터 엔지니어입니다.
사용자가 입력한 검색 키워드를 분석하여 표준 대표 명칭과 동의어를 JSON으로 정규화하세요.

입력 키워드: "{raw}"

지침:
1. canonical: 공식 표준 명칭 (구단이면 정식명칭, 선수면 선수명, 주제면 표준용어)
2. safe_id: 공백 및 특수문자가 없는 파일/DB용 식별자
3. synonyms: 기사 검색 및 연관도 산출에 사용할 동의어/약칭/별칭 리스트 (최대 5개)
4. entity_type: "team", "player", "topic", "general" 중 하나

반드시 순수 JSON 형식만 한 줄로 출력하세요:
{{"canonical": "...", "safe_id": "...", "synonyms": [...], "entity_type": "..."}}"""
                resp = self.client.responses.create(
                    model=self.model,
                    instructions="You are a strict KBO baseball entity normalizer. Output pure JSON only without markdown.",
                    input=prompt,
                )
                text = resp.output_text.strip()
                if text.startswith("```"):
                    text = text.split("```")[1]
                    if text.startswith("json"):
                        text = text[4:]
                data = json.loads(text.strip())

                canonical = data.get("canonical", cleaned)
                safe_id = re.sub(r"[^\w가-힣0-9_-]", "", data.get("safe_id", canonical)).strip() or cleaned
                syns = list(set([raw, cleaned, canonical] + data.get("synonyms", [])))
                etype = data.get("entity_type", "general")

                entity = NormalizedEntity(raw=raw, canonical=canonical, safe_id=safe_id, synonyms=syns, entity_type=etype)
                self._cache[raw] = entity
                return entity
            except Exception:
                pass

        # Fallback 기본값: 정제된 단어 사용
        safe_clean = re.sub(r"[^\w가-힣0-9_-]", "", cleaned).strip() or "야구"
        entity = NormalizedEntity(
            raw=raw,
            canonical=cleaned,
            safe_id=safe_clean,
            synonyms=list(set([raw, cleaned])),
            entity_type="general",
        )
        self._cache[raw] = entity
        return entity

print('KeywordNormalizer 클래스 정의 완료!')

KeywordNormalizer 클래스 정의 완료!


### 7.1. 다양한 형태의 키워드 정규화 테스트
- 구단 약칭, 띄어쓰기 오타, 선수명 조사 결합 등 실제 사용자 입력 패턴을 넣어 정규화 결과를 확인합니다.

In [3]:
# 정규화 엔진 인스턴스 생성
normalizer = KeywordNormalizer(model="gpt-5.6-luna")

# 테스트할 키워드 목록 (다양한 표기 변형)
test_keywords = [
    "한화",
    "한화이글스",
    "한화 이글스",
    "이글스",
    "기아",
    "KIA",
    "KIA 타이거즈",
    "엘지",
    "LG 트윈스",
    "쓱",
    "SSG 랜더스",
    "엔씨",
    "김 도 영",
    "김도영의",
    "가을야구",
]

results = []
for kw in test_keywords:
    res = normalizer.normalize(kw, use_llm_fallback=False)  # 1차 사전/규칙 테스트
    results.append({
        "입력 키워드": res.raw,
        "표준 공식명 (canonical)": res.canonical,
        "DB/파일명 ID (safe_id)": res.safe_id,
        "분류": res.entity_type,
        "동의어 풀 (synonyms)": ", ".join(res.synonyms),
    })

df_norm = pd.DataFrame(results)
print("=== [정규화 엔진 테스트 결과] ===")
df_norm

=== [정규화 엔진 테스트 결과] ===


,입력 키워드,표준 공식명 (canonical),DB/파일명 ID (safe_id),분류,동의어 풀 (synonyms)
0,한화,한화 이글스,한화이글스,team,"한화, 한화이글스, 이글스, 한화 이글스"
1,한화이글스,한화 이글스,한화이글스,team,"한화, 한화이글스, 이글스, 한화 이글스"
2,한화 이글스,한화 이글스,한화이글스,team,"한화, 한화이글스, 이글스, 한화 이글스"
3,이글스,한화 이글스,한화이글스,team,"한화, 한화이글스, 이글스, 한화 이글스"
4,기아,KIA 타이거즈,KIA타이거즈,team,"KIA, 기아, 타이거즈, KIA 타이거즈, 기아 타이거즈, KIA타이거즈, 기아타이거즈"
5,KIA,KIA 타이거즈,KIA타이거즈,team,"KIA, 기아, 타이거즈, KIA 타이거즈, 기아 타이거즈, KIA타이거즈, 기아타이거즈"
6,KIA 타이거즈,KIA 타이거즈,KIA타이거즈,team,"KIA, 기아, 타이거즈, KIA 타이거즈, 기아 타이거즈, KIA타이거즈, 기아타이거즈"
7,엘지,LG 트윈스,LG트윈스,team,"LG트윈스, 트윈스, LG 트윈스, 엘지 트윈스, 엘지, LG, 엘지트윈스"
8,LG 트윈스,LG 트윈스,LG트윈스,team,"LG트윈스, 트윈스, LG 트윈스, 엘지 트윈스, 엘지, LG, 엘지트윈스"
9,쓱,SSG 랜더스,SSG랜더스,team,"SSG, SSG랜더스, SSG 랜더스, 랜더스, SK 와이번스, SK와이번스, 쓱"


### 7.2. 보고서 핵심 기사 발췌 시 연관도 점수(`get_relevance_score`) 비교 실습
- **기존 방식**: 사용자가 '한화 이글스'로 검색했을 때, 기사 제목/본문에 '한화'라고만 적혀 있으면 점수가 0점이 되어 핵심 기사에서 탈락하는 문제 발생
- **신규 방식**: 정규화된 `synonyms` 풀 전체를 대조하여 모든 약칭/동의어 출현 빈도를 점수에 합산

In [4]:
# 가상의 기사 샘플 데이터
sample_articles = [
    {
        "title": "한화, 3연승 질주... 불펜 완벽 계투로 승리 견인",
        "content": "대전 경기에서 한화는 8회 터진 결승타로 극적인 승리를 거두었다. 이글스 팬들은 열광했다."
    },
    {
        "title": "KIA 타이거즈 김도영, 시즌 30호 홈런 작렬 대기록",
        "content": "광주 경기에서 기아의 중심 타자 김도영이 호쾌한 타격으로 승리를 이끌었다."
    },
    {
        "title": "LG 트윈스 가을야구 청신호, 잠실서 두산 격파",
        "content": "엘지 트윈스가 완벽한 투타 조화로 두산 베어스를 꺾고 상위권을 수성했다."
    }
]

def old_relevance_score(art: dict, keyword: str) -> int:
    """기존 단일 문자열 일치 방식"""
    t = art.get("title", "")
    c = art.get("content", "")
    return (t.count(keyword) * 15) + (c.count(keyword) * 3)

def new_relevance_score(art: dict, entity: NormalizedEntity) -> int:
    """신규 동의어 풀(Synonyms) 가중치 합산 방식"""
    t = art.get("title", "")
    c = art.get("content", "")
    score = 0
    # 중복 가산을 방지하기 위해 각 동의어의 출현 빈도를 합산
    for syn in entity.synonyms:
        if not syn:
            continue
        score += t.count(syn) * 15
        score += c.count(syn) * 3
    return score

# 사용자가 '한화 이글스'로 검색한 시나리오 테스트
search_query = "한화 이글스"
entity = normalizer.normalize(search_query)

print(f"[테스트 쿼리]: '{search_query}' (정규화 표준: '{entity.canonical}', 동의어: {entity.synonyms})\n")

comparison = []
for idx, art in enumerate(sample_articles, 1):
    old_score = old_relevance_score(art, search_query)
    new_score = new_relevance_score(art, entity)
    comparison.append({
        "기사 제목": art["title"],
        "기존 점수 (단일 일치)": old_score,
        "신규 점수 (동의어 풀)": new_score,
        "판정": "정상 발췌 (개선)" if old_score == 0 and new_score > 0 else "일치"
    })

df_comp = pd.DataFrame(comparison)
df_comp

[테스트 쿼리]: '한화 이글스' (정규화 표준: '한화 이글스', 동의어: ['한화', '한화이글스', '이글스', '한화 이글스'])



,기사 제목,기존 점수 (단일 일치),신규 점수 (동의어 풀),판정
0,"한화, 3연승 질주... 불펜 완벽 계투로 승리 견인",0,21,정상 발췌 (개선)
1,"KIA 타이거즈 김도영, 시즌 30호 홈런 작렬 대기록",0,0,일치
2,"LG 트윈스 가을야구 청신호, 잠실서 두산 격파",0,0,일치


### 7.3. DB 파일명 일원화 및 정규화 데이터 적재 시뮬레이션
- 사용자가 `한화`, `한화이글스`, `이글스` 중 어떤 것으로 수집하더라도 `safe_id`를 기반으로 `한화이글스_데이터베이스.db` 단일 파일에 누적 적재되는 로직을 검증합니다.

In [5]:
# DB 파일명 결정 로직 시뮬레이션
user_inputs = ["한화", "한화이글스", "한화 이글스", "기아", "KIA", "KIA 타이거즈"]

db_mapping_results = []
for u_in in user_inputs:
    ent = normalizer.normalize(u_in)
    target_db_filename = f"{ent.safe_id}_데이터베이스.db"
    target_report_filename = f"{ent.safe_id}_보고서_260911.md"
    
    db_mapping_results.append({
        "사용자 입력 키워드": u_in,
        "정규화 대표 공식명": ent.canonical,
        "일원화된 DB 파일명": target_db_filename,
        "일원화된 리포트 파일명": target_report_filename,
    })

df_db_map = pd.DataFrame(db_mapping_results)
print("=== [DB 및 리포트 파일명 일원화 매핑 결과] ===")
print("-> '한화', '한화이글스', '한화 이글스' 모두 동일한 단일 DB 파일로 매핑되어 누적됩니다.")
df_db_map

=== [DB 및 리포트 파일명 일원화 매핑 결과] ===
-> '한화', '한화이글스', '한화 이글스' 모두 동일한 단일 DB 파일로 매핑되어 누적됩니다.


,사용자 입력 키워드,정규화 대표 공식명,일원화된 DB 파일명,일원화된 리포트 파일명
0,한화,한화 이글스,한화이글스_데이터베이스.db,한화이글스_보고서_260911.md
1,한화이글스,한화 이글스,한화이글스_데이터베이스.db,한화이글스_보고서_260911.md
2,한화 이글스,한화 이글스,한화이글스_데이터베이스.db,한화이글스_보고서_260911.md
3,기아,KIA 타이거즈,KIA타이거즈_데이터베이스.db,KIA타이거즈_보고서_260911.md
4,KIA,KIA 타이거즈,KIA타이거즈_데이터베이스.db,KIA타이거즈_보고서_260911.md
5,KIA 타이거즈,KIA 타이거즈,KIA타이거즈_데이터베이스.db,KIA타이거즈_보고서_260911.md
